# Módulo 02 · Dominancia
**Teoría de Juegos — Tutorial Interactivo**

La eliminación iterada de estrategias estrictamente dominadas (EIED) es la primera herramienta de solución: si una estrategia es **siempre** peor que otra, ningún agente racional la jugaría.

In [ ]:
import numpy as np
from itertools import product
print('Entorno listo')

## 1. Representar un juego como par de matrices

Usamos dos matrices numpy: `A` para los pagos del Jugador 1, `B` para los del Jugador 2.
Cada fila es una estrategia de J1, cada columna una de J2.

In [ ]:
# Dilema del Prisionero
# Filas/cols: [Cooperar, Traicionar]
A = np.array([[3, 0],
              [5, 1]])  # pagos J1

B = np.array([[3, 5],
              [0, 1]])  # pagos J2

print('Matriz A (J1):')
print(A)
print('\nMatriz B (J2):')
print(B)
print('\nPago en (Traicionar, Cooperar): J1 =', A[1,0], ', J2 =', B[1,0])

## 2. Verificar dominancia estricta

La estrategia `r` domina estrictamente a `r2` para J1 si `A[r,c] > A[r2,c]` para **toda** columna `c`.

In [ ]:
def is_strictly_dominated_row(A, row_idx, active_rows, active_cols):
    """True si row_idx está estrictamente dominada por alguna otra fila activa."""
    sub = A[np.ix_(active_rows, active_cols)]
    local = list(active_rows).index(row_idx)
    for r in range(len(active_rows)):
        if r == local:
            continue
        if np.all(sub[r] > sub[local]):
            return True, active_rows[r]
    return False, None

def is_strictly_dominated_col(B, col_idx, active_rows, active_cols):
    """True si col_idx está estrictamente dominada para J2."""
    sub = B[np.ix_(active_rows, active_cols)]
    local = list(active_cols).index(col_idx)
    for c in range(len(active_cols)):
        if c == local:
            continue
        if np.all(sub[:, c] > sub[:, local]):
            return True, active_cols[c]
    return False, None

# Test en el Dilema del Prisionero
ar, ac = [0,1], [0,1]
dom, by = is_strictly_dominated_row(A, 0, ar, ac)
print(f'Cooperar (J1) dominada: {dom}', f'— por estrategia {by}' if dom else '')

## 3. Algoritmo EIED completo

In [ ]:
def iterated_elimination(A, B, row_labels=None, col_labels=None):
    rows, cols = A.shape
    active_rows = list(range(rows))
    active_cols = list(range(cols))
    rl = row_labels or [f'F{i}' for i in range(rows)]
    cl = col_labels or [f'C{j}' for j in range(cols)]
    steps = []
    changed = True
    
    while changed:
        changed = False
        
        for r in active_rows[:]:
            dom, by = is_strictly_dominated_row(A, r, active_rows, active_cols)
            if dom:
                steps.append(f'  → Eliminar fila "{rl[r]}" de J1 (dominada por "{rl[by]}")')
                active_rows.remove(r)
                changed = True
        
        for c in active_cols[:]:
            dom, by = is_strictly_dominated_col(B, c, active_rows, active_cols)
            if dom:
                steps.append(f'  → Eliminar col  "{cl[c]}" de J2 (dominada por "{cl[by]}")')
                active_cols.remove(c)
                changed = True
    
    return active_rows, active_cols, steps

# Aplicar al Dilema del Prisionero
rl = ['Cooperar', 'Traicionar']
cl = ['Cooperar', 'Traicionar']
ar, ac, steps = iterated_elimination(A, B, rl, cl)

print('=== EIED — Dilema del Prisionero ===')
for s in steps:
    print(s)
print(f'\nJuego reducido: filas {[rl[r] for r in ar]} × cols {[cl[c] for c in ac]}')
if len(ar) == 1 and len(ac) == 1:
    print(f'✓ Dominancia-soluble: equilibrio único = ({rl[ar[0]]}, {cl[ac[0]]})')

## 4. Ejemplo 3×3 con múltiples rondas de eliminación

In [ ]:
A3 = np.array([[4, 3, 1],
               [2, 5, 3],
               [1, 2, 4]])

B3 = np.array([[3, 2, 4],
               [1, 4, 2],
               [5, 1, 3]])

rl3 = ['Alto', 'Medio', 'Bajo']
cl3 = ['Izq', 'Centro', 'Der']

ar3, ac3, steps3 = iterated_elimination(A3, B3, rl3, cl3)
print('=== EIED — Juego 3×3 ===')
for s in steps3:
    print(s)
print(f'\nResultado: {[rl3[r] for r in ar3]} × {[cl3[c] for c in ac3]}')

# Mostrar el subjuego superviviente
sub_A = A3[np.ix_(ar3, ac3)]
sub_B = B3[np.ix_(ar3, ac3)]
print('\nPagos supervivientes:')
print('A:', sub_A)
print('B:', sub_B)

## 5. ¿Qué pasa con la dominancia débil?

La dominancia débil (≥ en todos los casos, > en alguno) no garantiza invarianza del orden de eliminación. Aquí vemos un ejemplo donde importa el orden.

In [ ]:
# Juego con dominancia débil
# Fila 0 domina débilmente a Fila 1 si A[0,:] >= A[1,:] con al menos un >
A_weak = np.array([[2, 2],
                   [2, 0]])  # Fila 0 domina débilmente a Fila 1
B_weak = np.array([[0, 2],
                   [0, 1]])

# Con EIED estricta, Fila 1 NO se elimina (no hay dominancia estricta)
# Pero si se permite débil, el resultado puede variar según el orden
print('Nota: la EIED con dominancia débil puede producir resultados diferentes')
print('según el orden de eliminación. La estricta es la que garantiza unicidad.')
print()
print('Matriz A:')
print(A_weak)
print('La fila 0 domina DÉBILMENTE a la fila 1 (igual en col 0, mejor en col 1)')

## 6. Ejercicios

### Ejercicio 1
Para el juego siguiente, identifica las estrategias dominadas e implementa un paso manual de EIED:
```
A = [[3, 1, 2], [4, 2, 1], [1, 3, 3]]
B = [[2, 4, 1], [1, 2, 3], [3, 1, 2]]
```

In [ ]:
# EJERCICIO 1 — completa el código
A_ej = np.array([[3, 1, 2], [4, 2, 1], [1, 3, 3]])
B_ej = np.array([[2, 4, 1], [1, 2, 3], [3, 1, 2]])

# TU CÓDIGO AQUÍ:
# 1. Aplica iterated_elimination(A_ej, B_ej)
# 2. ¿El juego es dominancia-soluble?
# 3. Si no, ¿qué necesitamos para resolver el juego restante?

# Test (descomenta al terminar):
# ar_ej, ac_ej, _ = iterated_elimination(A_ej, B_ej)
# assert len(ar_ej) >= 1 and len(ac_ej) >= 1  # al menos sobrevive algo

### Ejercicio 2
Demuestra que el **Dilema del Prisionero iterado** (2 rondas conocidas) sigue teniendo (Traicionar, Traicionar) como único equilibrio. Pista: aplica backward induction para reducirlo a un juego de una ronda.

In [ ]:
# EJERCICIO 2 — argumento de backward induction
# En la última ronda: ¿qué es lo racional?
# En la penúltima ronda: dado lo anterior, ¿qué es lo racional?
# TU CÓDIGO AQUÍ:
